# 02 — Evaluation harness noise floor

`structuring-ml-projects` SKILL.md step 4(b): measure the evaluation
noise floor empirically and early, before trusting any model delta.

Loads `data/processed/eda_features.csv` (written by
`01_eda_volumes.ipynb`'s handoff cell — run that first if this file
doesn't exist yet) and fits `abs_asym` alone across 5 fold-reshuffles
using `src/evaluate.py`'s real `make_folds` (joint target x site-proxy
stratification) and `log_loss_score` — the exact harness any future model
comparison will use.

**Note on data handling:** this notebook reads a *derived* feature file
(scalar columns only: `abs_asym`, other 6d features, `inplane_family`,
and the label) written by the EDA notebook — not `.nii.gz` files. It
still contains one row per patient with the label, so per the project's
AI-assistant data-handling rule this notebook's cells are still meant to
be run and inspected by you, not by Claude — only the aggregate
mean/std/per-seed log loss values should be shared back.

In [1]:
# [RUN ME]
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, str(Path.cwd().parent / "src"))
import config
import evaluate

feat_df = pd.read_csv(config.DATA_PROCESSED / "eda_features.csv")
print(f"{len(feat_df)} rows loaded")

N_SEEDS = 5
X = feat_df[["abs_asym"]].to_numpy(dtype=float)
y = feat_df[config.TARGET_COLUMN].to_numpy()
family = feat_df["inplane_family"].to_numpy()

seed_scores = []
for seed in range(N_SEEDS):
    folds = evaluate.make_folds(y, family, n_splits=config.N_FOLDS, random_state=seed)
    preds = np.zeros(len(y))
    for train_idx, test_idx in folds:
        model = LogisticRegression(max_iter=2000)
        model.fit(X[train_idx], y[train_idx])
        preds[test_idx] = model.predict_proba(X[test_idx])[:, 1]
    seed_scores.append(evaluate.log_loss_score(y, preds))

seed_scores = np.array(seed_scores)
print(f"\nlog loss across {N_SEEDS} fold-reshuffles (abs_asym only):")
print("individual seed scores:", np.round(seed_scores, 4).tolist())
print(f"mean={seed_scores.mean():.4f}  sd={seed_scores.std():.4f}")
print(f"\nvs. config.BASELINE_LOGLOSS={config.BASELINE_LOGLOSS:.4f}: "
      f"delta={seed_scores.mean() - config.BASELINE_LOGLOSS:+.4f}")


1362 rows loaded

log loss across 5 fold-reshuffles (abs_asym only):
individual seed scores: [0.6121, 0.6119, 0.6118, 0.6119, 0.6119]
mean=0.6119  sd=0.0001

vs. config.BASELINE_LOGLOSS=0.6884: delta=-0.0765


**What we're looking for:** the spread of log loss across fold-reshuffles
for a known-good single feature, as an empirical noise floor — any future
model's improvement smaller than this spread is not distinguishable from
partition noise.

**Why:** `structuring-ml-projects` SKILL.md step 4(b) — measure the noise
floor empirically and early; without it "is this delta real?" is a gut
call, not a comparison.

**Source:** `structuring-ml-projects` SKILL.md step 4.

**What we found** (run 2026-09-08, all 1362 rows, `evaluate.make_folds` +
`evaluate.log_loss_score`, `abs_asym` alone via `LogisticRegression`):
seed scores `[0.6121, 0.6119, 0.6118, 0.6119, 0.6119]`, **mean=0.6119,
sd=0.0001**. Matches section 6d's single-split result (0.6126) closely,
confirming `evaluate.py` reproduces that earlier ad hoc computation.
`delta = -0.0765` vs. `config.BASELINE_LOGLOSS = 0.6884`.

**This is a very tight floor, and that's expected, not a red flag** — it
reflects a low-variance probe: one feature with a strong, stable
relationship to the target (AUC 0.73, section 6d), a 2-parameter model
(slope + intercept), n=1362, and folds stratified jointly on target and
site proxy (which reduces between-fold variance further, that's what
`make_folds` is for). Different fold splits barely move a logistic fit
that simple and that well-supported by data.

**Decision / next step — important caveat, do not over-generalize this
number**: `sd=0.0001` is the noise floor for *this specific probe*
(1 feature, this model class, this fold design) — **not** a universal
threshold. A richer classical model (more features, regularization
choices) or the 3D-CNN (many more parameters, stochastic training,
augmentation) will very likely show more fold-to-fold variance and needs
**its own noise-floor measurement once built**, per `SKILL.md`: *"Re-measure
the floor whenever the fold structure, data, or model class changes."*
Use this run to confirm `evaluate.py`'s harness itself works correctly
end-to-end on real data (it does — the numbers match section 6d), not as
the final word on how much future deltas need to clear.